In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

cloud_file = next(Path("data").rglob("ireland_clouds_*.npz"))
stations_folder = Path("data/raw/stations")

print("Cloud file:", cloud_file)
print("Cloud file found:", cloud_file.exists())
print("Stations folder found:", stations_folder.exists())
print("Station CSV files:", len(list(stations_folder.glob("*.csv"))))

Cloud file: data\raw\satellite\ireland_clouds_20240101_20250101.npz
Cloud file found: True
Stations folder found: True
Station CSV files: 5


In [2]:
with np.load(cloud_file) as cloud_data:
    cloud_times = pd.to_datetime(cloud_data["times"], utc=True)

    print("Cloud timestamps:", len(cloud_times))
    print("Cloud mask shape:", cloud_data["cloud_mask"].shape)
    print("Cloud-top height shape:", cloud_data["cth_km"].shape)

print("First hour:", cloud_times[0])
print("Last hour:", cloud_times[-1])

Cloud timestamps: 8784
Cloud mask shape: (8784, 90, 120)
Cloud-top height shape: (8784, 90, 120)
First hour: 2024-01-01 00:00:00+00:00
Last hour: 2024-12-31 23:00:00+00:00


In [3]:
station_files = {
    "Casement": "hly3723_subset.csv",
    "Cork": "hly3904_subset.csv",
    "Shannon": "hly518_subset.csv",
    "Knock": "hly4935_subset.csv",
    "Dublin": "hly532.csv",
}

stations = {}

for name, filename in station_files.items():
    path = stations_folder / filename

    data = pd.read_csv(
        path,
        skiprows=41,
        usecols=["date", "rain", "ind", "temp", "rhum"],
        low_memory=False
    )

    data["date"] = pd.to_datetime(
        data["date"],
        format="%d-%b-%Y %H:%M",
        utc=True
    )

    data = data[data["date"].dt.year == 2024]
    stations[name] = data

    print(name, ":", len(data), "hours")

Casement : 8784 hours
Cork : 8784 hours
Shannon : 8784 hours
Knock : 8784 hours
Dublin : 8784 hours


In [4]:
combined = pd.DataFrame({"date": cloud_times})

for name, data in stations.items():
    station_columns = data[["date", "rain", "temp", "rhum", "ind"]].copy()

    station_columns = station_columns.rename(columns={
        "rain": f"{name}_rain",
        "temp": f"{name}_temp",
        "rhum": f"{name}_rhum",
        "ind": f"{name}_rain_ind",
    })

    combined = combined.merge(
        station_columns,
        on="date",
        how="left",
        validate="one_to_one"
    )

print("Combined shape:", combined.shape)
display(combined.head())

Combined shape: (8784, 21)


,date,Casement_rain,Casement_temp,Casement_rhum,Casement_rain_ind,Cork_rain,Cork_temp,Cork_rhum,Cork_rain_ind,Shannon_rain,...,Shannon_rhum,Shannon_rain_ind,Knock_rain,Knock_temp,Knock_rhum,Knock_rain_ind,Dublin_rain,Dublin_temp,Dublin_rhum,Dublin_rain_ind
0,2024-01-01 00:00:00+00:00,0.0,5.5,87,111,0.0,4.8,79,111,0.0,...,81,111,0.3,4.2,92,111,0.0,5.7,85,111
1,2024-01-01 01:00:00+00:00,0.0,5.0,86,111,0.0,4.8,78,111,0.0,...,76,111,0.0,3.7,92,111,0.0,5.2,86,111
2,2024-01-01 02:00:00+00:00,0.0,4.6,85,111,0.0,3.5,86,111,0.0,...,79,111,0.0,3.0,95,111,0.0,4.7,84,111
3,2024-01-01 03:00:00+00:00,0.0,4.1,87,111,0.0,4.1,87,111,0.0,...,75,111,0.2,2.5,95,111,0.0,4.5,84,111
4,2024-01-01 04:00:00+00:00,0.0,4.6,84,111,0.0,4.1,88,111,0.0,...,74,111,0.3,3.4,88,111,0.0,4.5,84,111


In [5]:
print("Rows:", len(combined))
print("Missing values in combined table:", combined.isna().sum().sum())

Rows: 8784
Missing values in combined table: 0


In [6]:
# Example: hours 00:00–05:00 are inputs; 06:00 is the target
input_columns = ["date"] + [
    f"{name}_{variable}"
    for name in station_files
    for variable in ["temp", "rhum"]
]

rain_columns = [f"{name}_rain" for name in station_files]

print("INPUT: six hours of temperature and humidity")
display(combined.loc[0:5, input_columns])

print("Also used as input: cloud maps at these same six hours")
print(combined.loc[0:5, "date"].tolist())

print("\nTARGET: rainfall at the following hour")
display(combined.loc[[6], ["date"] + rain_columns])

INPUT: six hours of temperature and humidity


,date,Casement_temp,Casement_rhum,Cork_temp,Cork_rhum,Shannon_temp,Shannon_rhum,Knock_temp,Knock_rhum,Dublin_temp,Dublin_rhum
0,2024-01-01 00:00:00+00:00,5.5,87,4.8,79,6.7,81,4.2,92,5.7,85
1,2024-01-01 01:00:00+00:00,5.0,86,4.8,78,6.6,76,3.7,92,5.2,86
2,2024-01-01 02:00:00+00:00,4.6,85,3.5,86,6.3,79,3.0,95,4.7,84
3,2024-01-01 03:00:00+00:00,4.1,87,4.1,87,6.4,75,2.5,95,4.5,84
4,2024-01-01 04:00:00+00:00,4.6,84,4.1,88,6.1,74,3.4,88,4.5,84
5,2024-01-01 05:00:00+00:00,4.0,83,4.0,90,5.5,84,2.7,95,4.2,84


Also used as input: cloud maps at these same six hours
[Timestamp('2024-01-01 00:00:00+0000', tz='UTC'), Timestamp('2024-01-01 01:00:00+0000', tz='UTC'), Timestamp('2024-01-01 02:00:00+0000', tz='UTC'), Timestamp('2024-01-01 03:00:00+0000', tz='UTC'), Timestamp('2024-01-01 04:00:00+0000', tz='UTC'), Timestamp('2024-01-01 05:00:00+0000', tz='UTC')]

TARGET: rainfall at the following hour


,date,Casement_rain,Cork_rain,Shannon_rain,Knock_rain,Dublin_rain
6,2024-01-01 06:00:00+00:00,0.0,0.0,0.0,0.2,0.0


In [7]:
weather_columns = input_columns[1:]  # temperature and humidity columns
sequence_length = 6

image_positions = []
weather_sequences = []
rain_targets = []

for target_row in range(sequence_length, len(combined)):
    first_input_row = target_row - sequence_length

    # Positions of the six cloud images
    image_positions.append(
        list(range(first_input_row, target_row))
    )

    # Temperature and humidity during those six hours
    weather_sequences.append(
        combined.iloc[first_input_row:target_row][weather_columns].to_numpy()
    )

    # Rainfall at the following hour, for all five stations
    rain_targets.append(
        combined.iloc[target_row][rain_columns].to_numpy()
    )

image_positions = np.array(image_positions)
weather_sequences = np.array(weather_sequences, dtype=np.float32)
rain_targets = np.array(rain_targets, dtype=np.float32)

print("Cloud image positions:", image_positions.shape)
print("Weather sequences:", weather_sequences.shape)
print("Rainfall targets:", rain_targets.shape)

Cloud image positions: (8778, 6)
Weather sequences: (8778, 6, 10)
Rainfall targets: (8778, 5)


In [8]:
example = 0
target_row = example + sequence_length

print("Input image times:")
for position in image_positions[example]:
    print(cloud_times[position])

print("\nRainfall target time:")
print(combined.loc[target_row, "date"])

print("\nRainfall at the five stations (mm):")
print(dict(zip(station_files.keys(), rain_targets[example])))

Input image times:
2024-01-01 00:00:00+00:00
2024-01-01 01:00:00+00:00
2024-01-01 02:00:00+00:00
2024-01-01 03:00:00+00:00
2024-01-01 04:00:00+00:00
2024-01-01 05:00:00+00:00

Rainfall target time:
2024-01-01 06:00:00+00:00

Rainfall at the five stations (mm):
{'Casement': np.float32(0.0), 'Cork': np.float32(0.0), 'Shannon': np.float32(0.0), 'Knock': np.float32(0.2), 'Dublin': np.float32(0.0)}


In [10]:
target_times = combined["date"].iloc[sequence_length:].reset_index(drop=True)

gap = sequence_length  # six hours between sets
usable_examples = len(target_times) - 2 * gap

train_count = round(0.70 * usable_examples)
validation_count = round(0.15 * usable_examples)

train_rows = np.arange(0, train_count)

validation_start = train_count + gap
validation_rows = np.arange(
    validation_start,
    validation_start + validation_count
)

test_start = validation_rows[-1] + 1 + gap
test_rows = np.arange(test_start, len(target_times))

total_used = len(train_rows) + len(validation_rows) + len(test_rows)

for name, rows in [
    ("Training", train_rows),
    ("Validation", validation_rows),
    ("Test", test_rows),
]:
    percentage = 100 * len(rows) / total_used
    print(
        f"{name}: {len(rows)} examples ({percentage:.1f}%) "
        f"| {target_times.iloc[rows[0]]} to {target_times.iloc[rows[-1]]}"
    )

Training: 6136 examples (70.0%) | 2024-01-01 06:00:00+00:00 to 2024-09-12 21:00:00+00:00
Validation: 1315 examples (15.0%) | 2024-09-13 04:00:00+00:00 to 2024-11-06 22:00:00+00:00
Test: 1315 examples (15.0%) | 2024-11-07 05:00:00+00:00 to 2024-12-31 23:00:00+00:00


**Scale temperature and humidity so their different units do not dominate each other**

In [11]:
# Calculate one mean and standard deviation for each of the 10 weather columns
weather_mean = weather_sequences[train_rows].mean(axis=(0, 1))
weather_std = weather_sequences[train_rows].std(axis=(0, 1))

# Prevent division by zero if a column never changes
weather_std[weather_std == 0] = 1

weather_scaled = (weather_sequences - weather_mean) / weather_std

print("Original weather shape:", weather_sequences.shape)
print("Scaled weather shape:", weather_scaled.shape)

print("\nFirst six hours, Casement temperature before scaling:")
print(weather_sequences[0, :, 0])

print("\nThe same values after scaling:")
print(weather_scaled[0, :, 0])

Original weather shape: (8778, 6, 10)
Scaled weather shape: (8778, 6, 10)

First six hours, Casement temperature before scaling:
[5.5 5.  4.6 4.1 4.6 4. ]

The same values after scaling:
[-1.092599  -1.1941681 -1.2754235 -1.3769926 -1.2754235 -1.3973064]


In [12]:
# Load the six cloud masks for our first example
with np.load(cloud_file) as cloud_data:
    six_masks = cloud_data["cloud_mask"][image_positions[0]]

# Channel 1: cloud present
cloud_channel = (six_masks == 2).astype(np.float32)

# Channel 2: valid observation (mask code 3 means no data)
valid_channel = (six_masks != 3).astype(np.float32)

# Put the two channels together for the CNN
cloud_example = np.stack(
    [cloud_channel, valid_channel],
    axis=-1
)

print("Original mask shape:", six_masks.shape)
print("CNN input shape:", cloud_example.shape)
print("First image: cloudy pixels:", int(cloud_channel[0].sum()))
print("First image: no-data pixels:", int((valid_channel[0] == 0).sum()))

Original mask shape: (6, 90, 120)
CNN input shape: (6, 90, 120, 2)
First image: cloudy pixels: 6907
First image: no-data pixels: 1


In [13]:
processed_folder = Path("data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)

prepared_file = processed_folder / "training_sequences.npz"

np.savez_compressed(
    prepared_file,
    image_positions=image_positions,
    weather=weather_scaled,
    rain=rain_targets,
    train_rows=train_rows,
    validation_rows=validation_rows,
    test_rows=test_rows,
    weather_mean=weather_mean,
    weather_std=weather_std,
    station_names=np.array(list(station_files.keys())),
)

print("Saved:", prepared_file)

Saved: data\processed\training_sequences.npz
